# 03b · MLP

**Owner:** Pranav

Tunes a multilayer perceptron (`MLPClassifier`, at most 20 configurations) with the same stratified 5-fold CV and the same selection metric as the SVM (`03a`), analyses the best configuration on out-of-fold (OOF) predictions, and saves its OOF scores for threshold selection in `04_evaluation`.

Shared repo code: `src.data` (loading), `src.preprocessing` (features + pipeline), `src.validation` (folds, OOF scores, F2 threshold, metrics). No test data is used in this notebook.

**Outputs**
- `results/tuning/mlp_configs.csv`: every configuration with its CV results
- `results/scores/mlp_oof.csv`: OOF scores of the best MLP (`PatientID, y_true, score`)
- `results/models/mlp.joblib`: best pipeline refitted on the full training set
- `results/figures/models/mlp_*.png`: diagnostic figures

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV, learning_curve, cross_val_score
from sklearn.metrics import precision_recall_curve, average_precision_score, ConfusionMatrixDisplay

from src.data import load_train, SEED
from src.preprocessing import add_engineered_features, build_preprocessor
from src.validation import get_cv, oof_scores, save_oof, best_f2_threshold, classification_metrics

RESULTS = os.path.join("..", "results")
FIG_PATH = os.path.join(RESULTS, "figures", "models")
for sub in ["tuning", "models", "scores", os.path.join("figures", "models")]:
    os.makedirs(os.path.join(RESULTS, sub), exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_PATH, name), dpi=150, bbox_inches="tight")
    plt.show()

## 1. Data and features
Same training set, features and preprocessing as the SVM (`build_preprocessor(use_engineered=True, use_bins=True)`, see notebook 02): scaled continuous features and symptom counts, one-hot Ethnicity, binary features and EducationLevel passed through, plus 10 quantile bins for UPDRS, MoCA and FunctionalAssessment. The test set stays locked until thresholds are frozen in `04_evaluation`.

In [ ]:
X_train, y_train = load_train()
X_train = add_engineered_features(X_train)
print(X_train.shape, "| PD share:", round(y_train.mean(), 3))

## 2. Baseline
Predicting everyone as PD: the reference every model has to beat (identical to `03a`).

In [ ]:
p = y_train.mean()
baseline = {"model": "baseline (all PD)", "TP": int(y_train.sum()), "FN": 0,
            "FP": int((1 - y_train).sum()), "TN": 0, "recall": 1.0, "precision": p,
            "specificity": 0.0, "F2": 5 * p / (4 * p + 1)}
pd.DataFrame([baseline]).round(3)

## 3. Validation strategy
Identical to the SVM so the two models are directly comparable:
- **Stratified 5-fold CV** on the 1,684 training patients with the shared `get_cv()` folds (seed 42); the test set is never used.
- **No leakage:** scaling, one-hot encoding and the quantile bins are inside the `Pipeline` and refitted per fold (bin edges learned from the training folds only, without labels). Scaling matters for the MLP because gradient descent converges poorly when inputs are on very different scales.
- **Selection metric: average precision (AP)**, threshold-free, because the F2 threshold is chosen afterwards (same reasoning as in `03a`).

## 4. Hyperparameter search (20 of max 20 configurations)

| Hyperparameter | Values | Why |
|---|---|---|
| `hidden_layer_sizes` | (8,), (32,), (64,), (32, 16), (64, 32) | Model capacity: one small, medium or wide hidden layer, or two layers |
| `alpha` (L2 penalty) | 0.001, 0.01, 0.1, 1 | Regularisation strength, log-spaced from weak to strong, so the optimum can fall inside the range rather than at its edge; counters overfitting on 1,684 patients |

5 × 4 = **20 configurations**. Fixed for all configurations (not tuned, to stay within the budget):
- `activation="relu"`, `solver="adam"` with the default learning rate 0.001: standard, robust defaults for small tabular networks (Goodfellow, Bengio & Courville, 2016, *Deep Learning*; Kingma & Ba, 2015, *Adam*).
- `early_stopping=True` (10% of each training fold held out internally, stop after 20 epochs without improvement), `max_iter=1000`: limits overfitting and training time.
- `random_state=SEED`: reproducible weight initialisation (seed sensitivity is checked in section 6).

In [ ]:
pipe = Pipeline([
    ("prep", build_preprocessor(use_engineered=True, use_bins=True)),
    ("mlp", MLPClassifier(activation="relu", solver="adam", early_stopping=True,
                          n_iter_no_change=20, max_iter=1000, random_state=SEED)),
])
param_grid = {
    "mlp__hidden_layer_sizes": [(8,), (32,), (64,), (32, 16), (64, 32)],
    "mlp__alpha": [0.001, 0.01, 0.1, 1.0],
}

search = GridSearchCV(pipe, param_grid, cv=get_cv(), scoring="average_precision",
                      return_train_score=True, n_jobs=-1)
search.fit(X_train, y_train)
n_configs = len(search.cv_results_["params"])
assert n_configs <= 20, "Assignment limit: at most 20 MLP configurations"
print("Configurations evaluated:", n_configs)

## 5. Results of all configurations
Each configuration is trained on 4 folds and validated on the 5th, five times: mean (± std) validation AP, mean train AP and the gap between them (larger gap = more overfitting). Sorted by validation AP.

In [ ]:
res = pd.DataFrame(search.cv_results_)
configs = pd.DataFrame({
    "hidden_layers": res["param_mlp__hidden_layer_sizes"].astype(str),
    "alpha": res["param_mlp__alpha"].astype(float),
    "val_AP": res["mean_test_score"].round(4),
    "val_AP_std": res["std_test_score"].round(4),
    "train_AP": res["mean_train_score"].round(4),
})
configs["overfit_gap"] = (configs["train_AP"] - configs["val_AP"]).round(4)
configs["rank"] = res["rank_test_score"]
configs = configs.sort_values("rank")
configs.to_csv(os.path.join(RESULTS, "tuning", "mlp_configs.csv"), index=False)
configs

**Selected MLP:** the configuration with the highest mean validation AP.

In [ ]:
best_params = search.best_params_
best = configs.iloc[0]
print(f"BEST MLP: hidden_layers={best.hidden_layers}, alpha={best.alpha:g} "
      f"| validation AP = {best.val_AP:.3f} ± {best.val_AP_std:.3f}")

### 5.1 Hyperparameter landscape
Validation AP (left) and overfitting gap = train AP − validation AP (right) for every architecture × alpha combination. Larger networks and weaker regularisation (small alpha) are expected to overfit more.

In [ ]:
order = ["(8,)", "(32,)", "(64,)", "(32, 16)", "(64, 32)"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col, title, cmap in [(axes[0], "val_AP", "Validation AP", "viridis"),
                             (axes[1], "overfit_gap", "Overfitting gap (train − validation AP)", "magma_r")]:
    grid = configs.pivot(index="hidden_layers", columns="alpha", values=col).loc[order]
    sns.heatmap(grid, annot=True, fmt=".3f", cmap=cmap, ax=ax)
    ax.set_title(f"MLP: {title}"); ax.set_xlabel("alpha (L2 penalty)"); ax.set_ylabel("hidden layers")
savefig("mlp_heatmaps.png")

## 6. Robustness of the best configuration
**Seed sensitivity:** an MLP's result depends on its random weight initialisation. The best configuration is re-evaluated with 5 different seeds (same folds); a small spread means the selection is not a lucky draw. These re-runs do not count as extra configurations because the hyperparameters are unchanged.

In [ ]:
seed_scores = {}
for seed in [0, 1, 2, 3, 4]:
    est = pipe.set_params(**best_params, mlp__random_state=seed)
    seed_scores[seed] = cross_val_score(est, X_train, y_train, cv=get_cv(),
                                        scoring="average_precision", n_jobs=-1).mean()
pipe.set_params(mlp__random_state=SEED)
seeds = pd.Series(seed_scores, name="val_AP").round(4)
print(f"Validation AP over 5 seeds: mean {seeds.mean():.4f}, std {seeds.std():.4f}, "
      f"range {seeds.min():.4f}–{seeds.max():.4f}")
seeds.to_frame().T

## 7. Learning curve of the best configuration
How validation AP changes with the amount of training data: a large, persistent train–validation gap points to variance (more data would help); two low, converged curves point to bias.

In [ ]:
sizes, tr, va = learning_curve(
    pipe.set_params(**best_params), X_train, y_train,
    train_sizes=[0.1, 0.25, 0.5, 0.75, 1.0], cv=get_cv(),
    scoring="average_precision", n_jobs=-1)

plt.figure(figsize=(6, 4))
plt.plot(sizes, tr.mean(axis=1), marker="o", label="Train")
plt.plot(sizes, va.mean(axis=1), marker="o", label="Validation (CV)")
plt.fill_between(sizes, va.mean(axis=1) - va.std(axis=1), va.mean(axis=1) + va.std(axis=1), alpha=0.2)
plt.xlabel("Training patients per fold"); plt.ylabel("Average precision")
plt.title("MLP learning curve (best configuration)"); plt.legend()
savefig("mlp_learning_curve.png")
pd.DataFrame({"train_size": sizes, "train_AP": tr.mean(axis=1).round(4), "val_AP": va.mean(axis=1).round(4)})

## 8. Best MLP vs best SVM on out-of-fold predictions
**What the code does:** runs the best MLP through `oof_scores()` (same folds as the SVM), and loads the SVM's OOF scores saved by `03a` (`results/scores/svm_oof.csv`). Both are compared on the same patients, training data only.

- **Table:** each model with the F2 rule applied to its OOF scores (a *preview*; the official thresholds are frozen in `04_evaluation`), next to the all-PD baseline.
- **Precision–recall curve:** precision (y) against recall (x) for every possible threshold; higher curve = better ranking, area = AP. Dotted line = baseline precision (62% PD).

In [ ]:
mlp_oof = oof_scores(pipe.set_params(**best_params), X_train, y_train)
svm_file = os.path.join(RESULTS, "scores", "svm_oof.csv")
oof = {"MLP": mlp_oof}
if os.path.exists(svm_file):
    svm = pd.read_csv(svm_file, index_col="PatientID")["score"]
    oof["SVM (03a)"] = svm.loc[y_train.index]
else:
    print("svm_oof.csv not found: run 03a first to include the SVM in the comparison.")

rows = [baseline]
for name, s in oof.items():
    t, _ = best_f2_threshold(y_train, s)
    r = classification_metrics(y_train, s, t, f"{name} (OOF, best-F2 thr)")
    r.update({"AP": average_precision_score(y_train, s), "threshold": t})
    rows.append(r)
pd.DataFrame(rows).round(3)

In [ ]:
plt.figure(figsize=(6, 4.5))
for name, s in oof.items():
    prec, rec, _ = precision_recall_curve(y_train, s)
    plt.plot(rec, prec, label=f"{name} (AP={average_precision_score(y_train, s):.3f})")
plt.axhline(y_train.mean(), color="grey", ls=":", label="Baseline (PD share)")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision–recall, MLP vs SVM (OOF)"); plt.legend()
savefig("mlp_pr_curve.png")

## 9. Best MLP: scores, threshold and confusion matrix
**Score and threshold:** the MLP outputs a **predicted probability of PD** between 0 and 1 (`predict_proba`). Every patient with probability ≥ threshold is predicted PD; the F2 rule picks the threshold that maximises F2. Here it is applied to the OOF scores as a *preview*; the official threshold is frozen in `04_evaluation`.

The three panels below:
1. **Score distribution by class:** x = OOF predicted probability of PD, y = number of patients; blue = actually PD, orange = actually non-PD. Dashed line = best-F2 threshold; everyone to its right is predicted PD.
2. **Metrics vs threshold:** x = a candidate threshold, y = the metric's value if that threshold were used. Moving right flags fewer patients as PD: recall falls, specificity and precision rise; F2 peaks at the dashed line.
3. **Confusion matrix at the best-F2 threshold:** rows = true class, columns = predicted class.

In [ ]:
t_best, f2_best = best_f2_threshold(y_train, mlp_oof)
ths = np.quantile(mlp_oof, np.linspace(0.01, 0.99, 200))
sweep = pd.DataFrame([classification_metrics(y_train, mlp_oof, t) for t in ths]).assign(threshold=ths)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for cls, lab in [(1, "PD"), (0, "Non-PD")]:
    axes[0].hist(mlp_oof[y_train == cls], bins=40, alpha=0.6, label=lab)
axes[0].axvline(t_best, color="k", ls="--", label=f"best-F2 threshold ({t_best:.2f})")
axes[0].set_xlabel("MLP predicted probability of PD (OOF)"); axes[0].set_ylabel("Number of patients")
axes[0].set_title("Score distribution by class"); axes[0].legend()

for m in ["recall", "precision", "specificity", "F2"]:
    axes[1].plot(sweep.threshold, sweep[m], label=m)
axes[1].axvline(t_best, color="k", ls="--")
axes[1].set_xlabel("Threshold"); axes[1].set_title("Metrics vs threshold (OOF)"); axes[1].legend()

y_pred = (mlp_oof >= t_best).astype(int)   # preview only; threshold frozen in 04
ConfusionMatrixDisplay.from_predictions(y_train, y_pred, display_labels=["Non-PD", "PD"],
                                        cmap="Blues", ax=axes[2], colorbar=False)
axes[2].set_title("Confusion matrix at best-F2 threshold (OOF)")
savefig("mlp_threshold_analysis.png")
print(f"Best-F2 threshold (preview) = {t_best:.3f}  ->  F2 = {f2_best:.3f}")

### 9.1 What-if: trading a little F2 for specificity
Not used for the model (the F2 rule is fixed). It quantifies the trade-off visible in panel 2: the highest specificity reachable while giving up at most 0.01 F2. Input for the decision-support discussion (Section 6 of the report).

In [ ]:
near = sweep[sweep.F2 >= f2_best - 0.01].sort_values("specificity").tail(1)
whatif = pd.concat([
    pd.DataFrame([classification_metrics(y_train, mlp_oof, t_best)]).assign(threshold=t_best, scenario="best-F2 threshold (rule)"),
    near.assign(scenario="what-if: max specificity, F2 loss <= 0.01"),
])
whatif[["scenario", "threshold", "F2", "recall", "precision", "specificity", "TP", "FN", "FP", "TN"]].reset_index(drop=True).round(3)

## 10. Save OOF scores and the final model
`save_oof()` writes the best MLP's OOF scores in the agreed format for `04_evaluation`. The model is the whole pipeline (preprocessing + MLP) refitted on the full training set, so notebook 04 only needs `add_engineered_features(X_test)` before `predict_proba(...)[:, 1]`.

In [ ]:
print("Saved:", save_oof("mlp", y_train, mlp_oof))
joblib.dump(search.best_estimator_, os.path.join(RESULTS, "models", "mlp.joblib"))
print("Saved results/models/mlp.joblib")

## 11. Notes for the report
- Best architecture and alpha, and why (capacity vs regularisation, heatmaps):
- Seed stability of the best configuration:
- Over/underfitting (train vs validation AP, learning curve):
- MLP vs SVM on OOF (AP, F2, specificity):
- OOF F2 vs baseline F2; threshold trade-off:
- Limitation: same folds used for tuning and OOF scores, so OOF metrics are slightly optimistic; the test set gives the unbiased estimate.